In [ ]:
import pandas as pd
from datetime import timedelta
from sklearn.preprocessing import LabelEncoder

## Агрегация и feature engineering

In [ ]:
clients_df = pd.read_csv('./data/clients.csv')
transactions_df = pd.read_csv('./data/transactions.csv')
loans_df = pd.read_csv('./data/loans.csv')

In [ ]:
clients_df

In [ ]:
transactions_df['transaction_date'] = pd.to_datetime(transactions_df['transaction_date'])
loans_df['start_date'] = pd.to_datetime(loans_df['start_date'])
loans_df['end_date'] = pd.to_datetime(loans_df['end_date'])

In [ ]:
prediction_date = pd.to_datetime('2025-01-01')

window_30_days = prediction_date - timedelta(days=30)
window_60_days = prediction_date - timedelta(days=60)

In [ ]:
# 30 days agg

# Filter transactions and loans within 30 days
transactions_30_df = transactions_df[(transactions_df['transaction_date'] >= window_30_days) & (transactions_df['transaction_date'] < prediction_date)]
loans_30_df = loans_df[(loans_df['start_date'] < prediction_date) & ((loans_df['end_date'] >= window_30_days) | loans_df['end_date'].isnull())]

# Aggregate transaction features
transaction_features_30 = transactions_30_df.groupby('client_id').agg(
    total_transaction_amount=('transaction_amount', 'sum'),
    avg_transaction_amount=('transaction_amount', 'mean'),
    std_transaction_amount=('transaction_amount', 'std'),
    transaction_count=('transaction_id', 'count'),
    late_payment_ratio=('payment_status', lambda x: (x == 'late').sum() / len(x)),
    avg_delinquency_days=('delinquency_days', 'mean'),
    max_delinquency_days=('delinquency_days', 'max')
).reset_index()

# Aggregate loan features
loan_features_30 = loans_30_df.groupby('client_id').agg(
    total_loan_amount=('loan_amount', 'sum'),
    avg_loan_amount=('loan_amount', 'mean'),
    loan_count=('loan_id', 'count'),
    avg_interest_rate=('interest_rate', 'mean'),
    remaining_balance_ratio=('remaining_balance', lambda x: x.sum() / loans_30_df['loan_amount'].sum())
).reset_index()

# Encode categorical features
label_encoders = {}

label_encoders['gender'] = LabelEncoder()
clients_df['gender_encoded'] = label_encoders['gender'].fit_transform(clients_df['gender'])

label_encoders['employment_status'] = LabelEncoder()
clients_df['employment_status_encoded'] = label_encoders['employment_status'].fit_transform(clients_df['employment_status'])

label_encoders['education_level'] = LabelEncoder()
clients_df['education_level_encoded'] = label_encoders['education_level'].fit_transform(clients_df['education_level'])

label_encoders['marital_status'] = LabelEncoder()
clients_df['marital_status_encoded'] = label_encoders['marital_status'].fit_transform(clients_df['marital_status'])

# Select relevant client features
client_features = clients_df[['client_id', 'age', 'gender_encoded', 'employment_status_encoded',
                              'income', 'education_level_encoded', 'marital_status_encoded',
                              'number_of_dependents']]

# Calculate additional derived client features
client_features['income_to_dependents_ratio'] = client_features['income'] / (client_features['number_of_dependents'] + 1)
client_features.drop(columns=["income", "number_of_dependents"], inplace=True)

# Merge client features with aggregated transaction and loan features
features_30 = client_features.merge(transaction_features_30, on='client_id', how='left').merge(loan_features_30, on='client_id', how='left')

# features_30.fillna(0, inplace=True)

In [ ]:
features_30

In [ ]:
for column in features_30.columns:
    if column != "client_id" or "encoded" not in column:
        features_30[column] = features_30[column].fillna(features_30[column].median(skipna=True))

In [ ]:
features_30

In [ ]:
from sklearn.cluster import KMeans

X = features_30.drop(columns=["client_id"])

kmeans = KMeans(n_clusters=2, random_state=777)
labels = kmeans.fit_predict(X)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=777)

## Тест моделей из sklearn

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

models = {
    "Logistic Regression": LogisticRegression(max_iter=10000),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, random_state=42),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_proba)
    }
    results.append(metrics)

# Вывод результатов
print(f"{'Model':<20} | {'Accuracy':<10}  | {'Precision':<10}   | {'Recall':<10}  | {'F1':<10} | {'ROC AUC':<10}")
for res in results:
    print(f"{res['Model']:<20} | {res['Accuracy']:.4f}      | {res['Precision']:.4f}       | {res['Recall']:.4f}      | {res['F1 Score']:.4f}     | {res['ROC AUC']:.4f}")

## LightGBM классификатор

In [ ]:
import lightgbm as lgb

train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'num_leaves': 27,
    'learning_rate': 0.05,
    'feature_fraction': 0.9
}

# Train the model
gbm = lgb.train(params, train_data, num_boost_round=100, valid_sets=[test_data])

In [ ]:
from sklearn.metrics import roc_auc_score

y_pred_prob = gbm.predict(X_test, num_iteration=gbm.best_iteration)

roc_auc = roc_auc_score(y_test, y_pred_prob)
print(f'ROC AUC Score: {roc_auc}')

## Взаимодействие с mlflow

In [ ]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:8081")
mlflow.set_experiment("LightGBM Classifier MLFlow demo")

In [ ]:
import mlflow.lightgbm
from mlflow.models import infer_signature

with mlflow.start_run():

    mlflow.log_params(params)

    mlflow.log_metric("roc_auc", roc_auc)

    signature = infer_signature(X_train, gbm.predict(X_train))

    mlflow.lightgbm.log_model(
        lgb_model=gbm,
        artifact_path="model",
        registered_model_name="lightgbm_classifier_demo",
        signature=signature,
        input_example=X_train
    )

In [ ]:
model_name = "lightgbm_classifier_demo"
version = "latest"
model_uri = f"models:/{model_name}/{version}"
loaded_model = mlflow.pyfunc.load_model(model_uri)

predictions = loaded_model.predict(X)
predictions

In [17]:
X.columns

Index(['age', 'gender_encoded', 'employment_status_encoded',
       'education_level_encoded', 'marital_status_encoded',
       'income_to_dependents_ratio', 'total_transaction_amount',
       'avg_transaction_amount', 'std_transaction_amount', 'transaction_count',
       'late_payment_ratio', 'avg_delinquency_days', 'max_delinquency_days',
       'total_loan_amount', 'avg_loan_amount', 'loan_count',
       'avg_interest_rate', 'remaining_balance_ratio'],
      dtype='object')

## Pyspark pipeline

In [ ]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

os.environ['HADOOP_CONF_DIR'] = '/opt/homebrew/Cellar/hadoop/3.4.1/libexec/etc/hadoop'
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
# os.environ['PYSPARK_PYTHON'] = '/Users/simiron2/python_venvs/diploma_venv/bin/python3'
# os.environ['PYSPARK_DRIVER_PYTHON'] = '/Users/simiron2/python_venvs/diploma_venv/bin/python3'

spark = SparkSession.builder \
    .appName("TestHDFS") \
    .config("spark.log.level", "INFO") \
    .config("spark.ui.bindAddress", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .master("yarn") \
    .getOrCreate() \

hdfs_path = "http://localhost:9000/user/simiron2/"

In [ ]:
spark

In [ ]:
# generate data

import random
from datetime import date, timedelta, datetime

GENDERS = ['M', 'F', 'U']
EMPLOYMENT_STATUSES = ['employed', 'unemployed', 'self-employed', 'student', 'retired', 'unknown']
EDUCATION_LEVELS = ['none', 'primary', 'secondary', 'tertiary', 'postgraduate', 'unknown']
MARITAL_STATUSES = ['single', 'married', 'divorced', 'widowed', 'unknown']
TRANSACTION_TYPES = ['purchase', 'withdrawal', 'deposit', 'loan_payment', 'transfer', 'fee']
PAYMENT_STATUSES = ['on-time', 'late', 'missed', 'unknown']
LOAN_TYPES = ['mortgage', 'auto', 'personal', 'student', 'credit_card', 'business']
LOAN_STATUSES = ['active', 'closed', 'delinquent', 'default']

random.seed(datetime.now().timestamp())

def random_date(start_date, end_date):
    days_between = (end_date - start_date).days
    random_days = random.randint(0, days_between)
    return start_date + timedelta(days=random_days)

def generate_client_ids(num_clients):
    client_ids = [i+1 for i in range(num_clients)]
    random.shuffle(client_ids)
    return client_ids

def generate_clients(client_ids):
    clients = []
    for client_id in client_ids:
        age = random.randint(18, 80)
        gender = random.choice(GENDERS)
        employment_status = random.choice(EMPLOYMENT_STATUSES)
        income = round(random.uniform(10000, 200000), 2)
        education_level = random.choice(EDUCATION_LEVELS)
        marital_status = random.choice(MARITAL_STATUSES)
        number_of_dependents = random.randint(0, 5)
        clients.append([client_id, age, gender, employment_status, income, education_level, marital_status, number_of_dependents])
    return clients

def generate_transactions(num_transactions, client_ids):
    transactions = []
    for i in range(num_transactions):
        client_id = random.choice(client_ids)
        transaction_date = random_date(date(2010, 1, 1), date.today())
        transaction_amount = round(random.uniform(10, 10000), 2)
        transaction_type = random.choice(TRANSACTION_TYPES)
        loan_id = random.randint(1, 1000) if transaction_type == 'loan_payment' else None
        payment_status = random.choice(PAYMENT_STATUSES)
        delinquency_days = random.randint(0, 90) if payment_status == 'late' else None
        transactions.append([i+1, client_id, transaction_date, transaction_amount, transaction_type, loan_id, payment_status, delinquency_days])
    return transactions

def generate_loans(num_loans, client_ids):
    loans = []
    for i in range(num_loans):
        client_id = random.choice(client_ids)
        loan_amount = round(random.uniform(1000, 500000), 2)
        loan_type = random.choice(LOAN_TYPES)
        loan_term = random.randint(1, 30) if loan_type in ['mortgage', 'auto', 'personal'] else random.randint(1, 10)
        interest_rate = round(random.uniform(2, 25), 2)
        start_date = random_date(date(2010, 1, 1), date.today())
        end_date = start_date + timedelta(days=loan_term*365)
        remaining_balance = round(loan_amount * (1 - random.uniform(0, 1)), 2)
        loan_status = random.choice(LOAN_STATUSES)
        loans.append([i+1, client_id, loan_amount, loan_type, loan_term, interest_rate, start_date, end_date, remaining_balance, loan_status])
    return loans


num_clients = 10_000
num_transactions = 40_000
num_loans = 4000

client_ids = generate_client_ids(num_clients)

clients = generate_clients(client_ids)
transactions = generate_transactions(num_transactions, client_ids)
loans = generate_loans(num_loans, client_ids)

clients_schema = ['client_id', 'age', 'gender', 'employment_status', 'income', 'education_level', 'marital_status', 'number_of_dependents']
transactions_schema = ['transaction_id', 'client_id', 'transaction_date', 'transaction_amount', 'transaction_type', 'loan_id', 'payment_status', 'delinquency_days']
loans_schema = ['loan_id', 'client_id', 'loan_amount', 'loan_type', 'loan_term', 'interest_rate', 'start_date', 'end_date', 'remaining_balance', 'loan_status']

clinet_sdf = spark.createDataFrame(clients, schema=clients_schema)
transactions_sdf = spark.createDataFrame(transactions, transactions_schema)
loans_sdf = spark.createDataFrame(loans, loans_schema)

In [ ]:
loans_sdf.show(5, truncate=False)

In [ ]:
(
    clinet_sdf
    .withColumn("business_dt", F.lit(datetime.now().strftime("%Y-%m-%d")))
    .repartition(1)
    .write
    .partitionBy("business_dt")
    .orc("clents", mode="overwrite")
)

In [ ]:
(
    transactions_sdf
    .withColumn("business_dt", F.lit(datetime.now().strftime("%Y-%m-%d")))
    .repartition(1)
    .write
    .partitionBy("business_dt")
    .orc("transactions", mode="overwrite")
)

In [ ]:
(
    loans_sdf
    .withColumn("business_dt", F.lit(datetime.now().strftime("%Y-%m-%d")))
    .repartition(1)
    .write
    .partitionBy("business_dt")
    .orc("loans", mode="overwrite")
)

In [ ]:
spark.read.orc("transactions").show()

In [ ]:
# preparing data

from pyspark.sql.types import DateType
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

clinet_sdf = spark.read.orc("clents")
transactions_sdf = spark.read.orc("transactions")
loans_sdf = spark.read.orc("loans")

prediction_date_str = '2022-01-01'
pred_date_col = F.lit(prediction_date_str).cast(DateType())
window_30_days = F.date_sub(pred_date_col, 30)

transactions_30_df = transactions_sdf.filter(
    (F.col('transaction_date').cast(DateType()) >= window_30_days) &
    (F.col('transaction_date').cast(DateType()) < pred_date_col)
)

loans_30_df = loans_sdf.filter(
    (F.col('start_date').cast(DateType()) < pred_date_col) &
    ((F.col('end_date').cast(DateType()) >= window_30_days) | F.col('end_date').isNull())
)

transaction_features_30 = transactions_30_df.groupBy('client_id').agg(
    F.sum('transaction_amount').alias('total_transaction_amount'),
    F.mean('transaction_amount').alias('avg_transaction_amount'),
    F.stddev_samp('transaction_amount').alias('std_transaction_amount'),
    F.count('transaction_id').alias('transaction_count'),
    (F.sum(F.when(F.col('payment_status') == 'late', 1).otherwise(0)) / F.count('*')).alias('late_payment_ratio'),
    F.mean('delinquency_days').alias('avg_delinquency_days'),
    F.max('delinquency_days').alias('max_delinquency_days')
)

total_loan_amount_row = loans_30_df.select(F.sum('loan_amount').alias('total_loan_amount')).first()
total_loan_amount = total_loan_amount_row['total_loan_amount'] if total_loan_amount_row['total_loan_amount'] is not None else 0.0

safe_total_loan_amount = total_loan_amount if total_loan_amount != 0 else 1e-8

loan_features_30 = loans_30_df.groupBy('client_id').agg(
    F.sum('loan_amount').alias('total_loan_amount'),
    F.mean('loan_amount').alias('avg_loan_amount'),
    F.count('loan_id').alias('loan_count'),
    F.mean('interest_rate').alias('avg_interest_rate'),
    (F.sum('remaining_balance') / safe_total_loan_amount).alias('remaining_balance_ratio')
)

category_cols = ['gender', 'employment_status', 'education_level', 'marital_status']
indexers = [StringIndexer(inputCol=col, outputCol=f"{col}_encoded", handleInvalid="keep") for col in category_cols]


pipeline = Pipeline(stages=indexers)
clients_encoded_df = pipeline.fit(clinet_sdf).transform(clinet_sdf)

client_features = clients_encoded_df.select(
    'client_id',
    'age',
    'gender_encoded',
    'employment_status_encoded',
    'income',
    'education_level_encoded',
    'marital_status_encoded',
    'number_of_dependents'
)

client_features = client_features.withColumn(
    'income_to_dependents_ratio',
    F.col('income') / (F.col('number_of_dependents') + 1)
).drop('income', 'number_of_dependents')

features_30 = (
    client_features
    .join(
        transaction_features_30, 
        on='client_id', 
        how='left')
    .join(
        loan_features_30, 
        on='client_id', 
        how='left')
)

cols_to_fill = [
    col_name for col_name in features_30.columns
    if col_name != 'client_id' and not col_name.endswith('_encoded')
]

for col_name in cols_to_fill:
    try:
        median = features_30.approxQuantile(col_name, [0.5], 0.01)[0]
        if median is not None:
            features_30 = features_30.fillna({col_name: median})
    except Exception as e:
        print(f"Could not impute column '{col_name}': {str(e)}")

features_30.show()

In [ ]:
(
    features_30
    .withColumn("business_dt", F.lit(datetime.now().strftime("%Y-%m-%d")))
    .repartition(1)
    .write
    .partitionBy("business_dt")
    .orc("model_features", mode="overwrite")
)

In [42]:
spark.stop()

In [ ]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

os.environ['HADOOP_CONF_DIR'] = '/opt/homebrew/Cellar/hadoop/3.4.1/libexec/etc/hadoop'
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
os.environ['JAVA_HOME'] = '/opt/homebrew/Cellar/openjdk@11/11.0.26/libexec/openjdk.jdk/Contents/Home'


spark = SparkSession.builder \
    .appName("TestHDFS") \
    .config("spark.log.level", "WARN") \
    .config("spark.ui.bindAddress", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .master("yarn") \
    .getOrCreate() \

spark

https://mmlspark.azureedge.net/maven added as a remote repository with the name: repo-1
Ivy Default Cache set to: /Users/simiron2/.ivy2/cache
The jars for the packages stored in: /Users/simiron2/.ivy2/jars
com.microsoft.azure#synapseml_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-1916d670-b55d-40f4-a3f8-09bc0aea5ba9;1.0
	confs: [default]
	found com.microsoft.azure#synapseml_2.12;1.0.11 in central
	found com.microsoft.azure#synapseml-core_2.12;1.0.11 in central


:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/3.5.5/libexec/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.apache.spark#spark-avro_2.12;3.4.1 in central
	found org.tukaani#xz;1.9 in central
	found commons-lang#commons-lang;2.6 in central
	found org.scalactic#scalactic_2.12;3.2.14 in central
	found org.scala-lang#scala-reflect;2.12.15 in central
	found io.spray#spray-json_2.12;1.3.5 in central
	found com.jcraft#jsch;0.1.54 in central
	found org.apache.httpcomponents.client5#httpclient5;5.1.3 in central
	found org.apache.httpcomponents.core5#httpcore5;5.1.3 in central
	found org.apache.httpcomponents.core5#httpcore5-h2;5.1.3 in central
	found org.slf4j#slf4j-api;1.7.25 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpmime;4.5.13 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.2 in central
	found com.linkedin.isolation-forest#isolation-forest_3.4.2_2.12;3.0.4 in central
	found com.chuusai#shapeless_2.12;2.3.10

In [9]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans

features_30_df = spark.read.orc("model_features")

feature_cols = [
    'age', 'gender_encoded', 'employment_status_encoded',
    'education_level_encoded', 'marital_status_encoded',
    'income_to_dependents_ratio', 'total_transaction_amount',
    'avg_transaction_amount', 'std_transaction_amount', 'transaction_count',
    'late_payment_ratio', 'avg_delinquency_days', 'max_delinquency_days',
    'total_loan_amount', 'avg_loan_amount', 'loan_count',
    'avg_interest_rate', 'remaining_balance_ratio']

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

assembled_df = assembler.transform(features_30_df)

kmeans = KMeans(k=2, seed=777)
kmeans_model = kmeans.fit(assembled_df)

clustered_df = kmeans_model.transform(assembled_df).withColumnRenamed("prediction", "target")

clustered_df.show()

+---------+---+--------------+-------------------------+-----------------------+----------------------+--------------------------+------------------------+----------------------+----------------------+-----------------+------------------+--------------------+--------------------+-----------------+------------------+----------+-----------------+-----------------------+-----------+--------------------+------+
|client_id|age|gender_encoded|employment_status_encoded|education_level_encoded|marital_status_encoded|income_to_dependents_ratio|total_transaction_amount|avg_transaction_amount|std_transaction_amount|transaction_count|late_payment_ratio|avg_delinquency_days|max_delinquency_days|total_loan_amount|   avg_loan_amount|loan_count|avg_interest_rate|remaining_balance_ratio|business_dt|            features|target|
+---------+---+--------------+-------------------------+-----------------------+----------------------+--------------------------+------------------------+----------------------+

In [10]:
(
    clustered_df
    .drop('features', 'business_dt')
    .repartition(1)
    .write
    .orc("model_features_marked", mode="overwrite")
)

In [2]:
whole_train_df = spark.read.orc("model_features_marked")

In [3]:
whole_train_df.show()

+---------+---+--------------+-------------------------+-----------------------+----------------------+--------------------------+------------------------+----------------------+----------------------+-----------------+------------------+--------------------+--------------------+-----------------+------------------+----------+-----------------+-----------------------+------+
|client_id|age|gender_encoded|employment_status_encoded|education_level_encoded|marital_status_encoded|income_to_dependents_ratio|total_transaction_amount|avg_transaction_amount|std_transaction_amount|transaction_count|late_payment_ratio|avg_delinquency_days|max_delinquency_days|total_loan_amount|   avg_loan_amount|loan_count|avg_interest_rate|remaining_balance_ratio|target|
+---------+---+--------------+-------------------------+-----------------------+----------------------+--------------------------+------------------------+----------------------+----------------------+-----------------+------------------+------

In [4]:
(
    whole_train_df
    .groupBy("target").count().show()
)

+------+-----+
|target|count|
+------+-----+
|     1|  637|
|     0| 9363|
+------+-----+



In [14]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split

df = (
    whole_train_df
    .drop("client_id")
    .toPandas()
)

y = df[['target']]
X = df.drop(columns=['target'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=777)

train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'num_leaves': 27,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'is_unbalance': True
}

# Train the model
gbm = lgb.train(params, train_data, num_boost_round=100, valid_sets=[test_data])

[LightGBM] [Info] Number of positive: 526, number of negative: 7474
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001813 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1511
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.065750 -> initscore=-2.653884
[LightGBM] [Info] Start training from score -2.653884
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

In [15]:
from sklearn.metrics import roc_auc_score

y_pred_prob = gbm.predict(X_test, num_iteration=gbm.best_iteration)

roc_auc = roc_auc_score(y_test, y_pred_prob)
print(f'ROC AUC Score: {roc_auc}')

ROC AUC Score: 1.0


In [16]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:8081")
mlflow.set_experiment("LightGBM Classifier MLFlow demo + Spark")

import mlflow.lightgbm
from mlflow.models import infer_signature

with mlflow.start_run():

    mlflow.log_params(params)

    mlflow.log_metric("roc_auc", roc_auc)

    signature = infer_signature(X_train, gbm.predict(X_train))

    mlflow.lightgbm.log_model(
        lgb_model=gbm,
        artifact_path="model",
        registered_model_name="lightgbm_classifier_spark_demo",
        signature=signature,
        input_example=X_train
    )

2025/05/01 12:09:54 INFO mlflow.tracking.fluent: Experiment with name 'LightGBM Classifier MLFlow demo + Spark' does not exist. Creating a new experiment.
/Users/simiron2/python_venvs/diploma_venv/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Successfully registered model 'lightgbm_classifier_spark_demo'

🏃 View run caring-wolf-187 at: http://127.0.0.1:8081/#/experiments/128612331869750552/runs/f3ac678509864c18813966ecb26d2bf2
🧪 View experiment at: http://127.0.0.1:8081/#/experiments/128612331869750552


Created version '1' of model 'lightgbm_classifier_spark_demo'.


In [15]:
import mlflow
from pyspark.sql.types import DecimalType

mlflow.set_tracking_uri("http://127.0.0.1:8081")

model_name = "lightgbm_classifier_spark_demo"
version = "latest"
model_uri = f"models:/{model_name}/{version}"
model_udf = mlflow.pyfunc.spark_udf(spark, model_uri=model_uri)

features = ['age', 'gender_encoded', 'employment_status_encoded',
       'education_level_encoded', 'marital_status_encoded',
       'income_to_dependents_ratio', 'total_transaction_amount',
       'avg_transaction_amount', 'std_transaction_amount', 'transaction_count',
       'late_payment_ratio', 'avg_delinquency_days', 'max_delinquency_days',
       'total_loan_amount', 'avg_loan_amount', 'loan_count',
       'avg_interest_rate', 'remaining_balance_ratio']

test_sdf = spark.read.orc("model_features")

# Apply UDF to your PySpark DataFrame
predictions_spark = test_sdf.withColumn("target", model_udf(*features).getItem(0).cast(DecimalType(precision=15, scale=10)))

2025/05/01 13:15:43 WARNING mlflow.pyfunc: Calling `spark_udf()` with `env_manager="local"` does not recreate the same environment that was used during training, which may lead to errors or inaccurate predictions. We recommend specifying `env_manager="conda"`, which automatically recreates the environment that was used to train the model and performs inference in the recreated environment.
2025/05/01 13:15:43 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


In [17]:
predictions_spark.show(truncate=False)

+---------+---+--------------+-------------------------+-----------------------+----------------------+--------------------------+------------------------+----------------------+----------------------+-----------------+------------------+--------------------+--------------------+-----------------+------------------+----------+-----------------+-----------------------+-----------+------------+
|client_id|age|gender_encoded|employment_status_encoded|education_level_encoded|marital_status_encoded|income_to_dependents_ratio|total_transaction_amount|avg_transaction_amount|std_transaction_amount|transaction_count|late_payment_ratio|avg_delinquency_days|max_delinquency_days|total_loan_amount|avg_loan_amount   |loan_count|avg_interest_rate|remaining_balance_ratio|business_dt|target      |
+---------+---+--------------+-------------------------+-----------------------+----------------------+--------------------------+------------------------+----------------------+----------------------+-------

In [30]:
(
    predictions_spark
    .select("client_id", "business_dt", "target")
    .repartition(1)
    .write
    .partitionBy("business_dt")
    .orc("predicted_scores", mode="overwrite")
)

## DQ

In [33]:
model_features = spark.read.orc("model_features")
max_business_dt = model_features.select(F.max("business_dt")).collect()[0][0].strftime("%Y-%m-%d")
row_count = model_features.filter(F.col("business_dt")==max_business_dt).select("business_dt", "client_id").count()
{max_business_dt: row_count}

{'2025-04-30': 10000}

In [34]:
row_distinct_count = model_features.filter(F.col("business_dt")==max_business_dt).select("business_dt", "client_id").distinct().count()
{max_business_dt: row_distinct_count}

{'2025-04-30': 10000}

In [35]:
predicted_scores = spark.read.orc("predicted_scores")
predicted_scores.show()

+---------+------------+-----------+
|client_id|      target|business_dt|
+---------+------------+-----------+
|     9363|0.0004422654| 2025-04-30|
|     9046|0.0004422654| 2025-04-30|
|     5475|0.0004422654| 2025-04-30|
|     7786|0.0004422654| 2025-04-30|
|     2581|0.0004422651| 2025-04-30|
|     7542|0.0004422654| 2025-04-30|
|     5826|0.0004422654| 2025-04-30|
|     7802|0.0004422654| 2025-04-30|
|     7496|0.0004422341| 2025-04-30|
|     1481|0.0004422654| 2025-04-30|
|      886|0.0004422654| 2025-04-30|
|     1304|0.0004422654| 2025-04-30|
|     7197|0.0004422654| 2025-04-30|
|     2531|0.0004422654| 2025-04-30|
|     5014|0.0004422654| 2025-04-30|
|     8269|0.0004422654| 2025-04-30|
|     1475|0.0004422654| 2025-04-30|
|     9733|0.0004422654| 2025-04-30|
|      104|0.0004422654| 2025-04-30|
|     7173|0.0004422654| 2025-04-30|
+---------+------------+-----------+
only showing top 20 rows



In [38]:
max_scores_business_dt = predicted_scores.select(F.max("business_dt")).collect()[0][0].strftime("%Y-%m-%d")
count_all = predicted_scores.filter(F.col("business_dt")==max_scores_business_dt).select('target').count()
null_count = (
    predicted_scores
    .filter(F.col("business_dt")==max_scores_business_dt)
    .select('target')
    .filter(F.col("target").isNull())
    .count()
)
{max_scores_business_dt: (null_count/count_all, (count_all - null_count)/count_all)}

{'2025-04-30': (0.0, 1.0)}

In [40]:
features_metrics = [
    (max_business_dt, row_count, row_distinct_count)
]

scoring_metrics = [
    (max_scores_business_dt, null_count/count_all, (count_all - null_count)/count_all)
]

In [ ]:
import psycopg2
from psycopg2.extras import execute_values

try:
    connection = psycopg2.connect(
        dbname='scoring',
        user='postgres_dev',
        host='localhost',
        port='5432'
    )

    cursor = connection.cursor()
    
    insert_features_dq_query = "INSERT INTO features_dq_metrics (business_dt, count_all, count_unique) VALUES %s"
    execute_values(cursor, insert_features_dq_query, features_metrics)
    connection.commit()

    insert_scoring_dq_query = "INSERT INTO scores_dq_metrics (business_dt, count_null, count_not_null) VALUES %s"
    execute_values(cursor, insert_scoring_dq_query, scoring_metrics)
    connection.commit()

except Exception as error:
    print("Error while connecting to PostgreSQL", error)

finally:
    if connection:
        cursor.close()
        connection.close()
        print("PostgreSQL connection is closed")

PostgreSQL connection is closed


## Postgres scores

In [1]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

os.environ['HADOOP_CONF_DIR'] = '/opt/homebrew/Cellar/hadoop/3.4.1/libexec/etc/hadoop'
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
os.environ['JAVA_HOME'] = '/opt/homebrew/Cellar/openjdk@11/11.0.26/libexec/openjdk.jdk/Contents/Home'


spark = SparkSession.builder \
    .appName("TestHDFS") \
    .config("spark.log.level", "ERROR") \
    .config("spark.ui.bindAddress", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.2.18") \
    .master("yarn") \
    .getOrCreate() \

spark

:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/3.5.5/libexec/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/simiron2/.ivy2/cache
The jars for the packages stored in: /Users/simiron2/.ivy2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6c10ae58-58a9-465b-a87c-d064da252a22;1.0
	confs: [default]
	found org.postgresql#postgresql;42.2.18 in central
	found org.checkerframework#checker-qual;3.5.0 in central
downloading https://repo1.maven.org/maven2/org/postgresql/postgresql/42.2.18/postgresql-42.2.18.jar ...
	[SUCCESSFUL ] org.postgresql#postgresql;42.2.18!postgresql.jar (334ms)
downloading https://repo1.maven.org/maven2/org/checkerframework/checker-qual/3.5.0/checker-qual-3.5.0.jar ...
	[SUCCESSFUL ] org.checkerframework#checker-qual;3.5.0!checker-qual.jar (110ms)
:: resolution report :: resolve 608ms :: artifacts dl 448ms
	:: modules in use:
	org.checkerframework#checker-qual;3.5.0 from central in [default]
	org.postgresql#postgresql;42.2.18 from central in [default]
	------------------------

In [2]:
predicted_scores = spark.read.orc("predicted_scores")
predicted_scores.show()

+---------+------------+-----------+
|client_id|      target|business_dt|
+---------+------------+-----------+
|     9363|0.0004422654| 2025-04-30|
|     9046|0.0004422654| 2025-04-30|
|     5475|0.0004422654| 2025-04-30|
|     7786|0.0004422654| 2025-04-30|
|     2581|0.0004422651| 2025-04-30|
|     7542|0.0004422654| 2025-04-30|
|     5826|0.0004422654| 2025-04-30|
|     7802|0.0004422654| 2025-04-30|
|     7496|0.0004422341| 2025-04-30|
|     1481|0.0004422654| 2025-04-30|
|      886|0.0004422654| 2025-04-30|
|     1304|0.0004422654| 2025-04-30|
|     7197|0.0004422654| 2025-04-30|
|     2531|0.0004422654| 2025-04-30|
|     5014|0.0004422654| 2025-04-30|
|     8269|0.0004422654| 2025-04-30|
|     1475|0.0004422654| 2025-04-30|
|     9733|0.0004422654| 2025-04-30|
|      104|0.0004422654| 2025-04-30|
|     7173|0.0004422654| 2025-04-30|
+---------+------------+-----------+
only showing top 20 rows



In [3]:
db_url = "jdbc:postgresql://localhost:5432/scoring"
db_properties = {
    "user": "postgres_dev",
    "password": "",
    "driver": "org.postgresql.Driver"
}

(
    predicted_scores
    .select("client_id", "target")
    .withColumnRenamed("client_id", "id")
    .withColumn("id", F.col("id").cast("string"))
    .withColumnRenamed("target", "score")
    .write
    .jdbc(url=db_url, table="scores_service", mode="overwrite", properties=db_properties)
)

In [4]:
spark.stop()